1/Thư viện

In [6]:
import numpy as np
from skimage import data

2/clip_histogram

In [7]:
def clip_histogram(hist, Beta):
    """Cắt và phân phối lại lược đồ
    hist: mảng (L,) số nguyên. Trả về mảng (L,) đã cắt, tổng không đổi."""
    hist = np.asarray(hist, dtype=np.int64)
    beta = max(1, int(round(Beta)))
    L = len(hist)

    clipped = np.clip(hist, 0, beta)
    excess = int(np.sum(hist - clipped))

    hist = clipped + excess // L
    remain = excess % L
    hist[:remain] += 1
    return hist

3/build_tile_luts

In [8]:
def build_tile_luts(img, grid=(8, 8), clip_limit=2.0):
    """Trả về mảng (Gy, Gx, L) uint8: bảng tra của từng ô """
    assert img.dtype == np.uint8 and img.ndim == 2
    H, W = img.shape

    Gy, Gx = grid
    if Gy > H or Gx > W or Gy < 1 or Gx <1:
        raise ValueError("Lưới lớn hơn ảnh")
    
    L = 256
    
    luts = np.zeros((Gy, Gx, L), dtype=np.uint8)
    
    for i in range(Gy):
        for j in range(Gx):
            y_start, y_end = i * H // Gy, (i + 1) * H // Gy
            x_start, x_end = j * W // Gx, (j + 1) * W // Gx
            
            tile = img[y_start:y_end, x_start:x_end]
            N_tile = tile.size
            hist = np.bincount(tile.ravel(), minlength=L)

            # beta = N_tile/L * [1 + α/100 * (s_max - 1)] = N_tile/L * s_max)  (tương đương alpha=100, s_max=clip_limit trong công thức)
            beta = N_tile / L * clip_limit
            clipped_hist = clip_histogram(hist, beta)
            
            cdf = np.cumsum(clipped_hist)
            luts[i, j] = np.round((L - 1) * cdf / cdf[-1]).astype(np.uint8)
            
    return luts

4/Clahe

In [9]:
def clahe(img, grid=(8, 8), clip_limit=2.0):
    """CLAHE đầy đủ. img: (H, W) uint8. Trả về (H, W) uint8 """
    assert img.dtype == np.uint8 and img.ndim == 2
    H, W = img.shape
    Gy, Gx = grid
    luts = build_tile_luts(img, grid, clip_limit)
    
    out = np.zeros_like(img, dtype=np.float64)
    tile_h = H / Gy
    tile_w = W / Gx
    
    for y in range(H):
        for x in range(W):
            fy = (y + 0.5) / tile_h - 0.5
            fx = (x + 0.5) / tile_w - 0.5
            
            i = int(fy//1)
            j = int(fx//1)
            
            a = fx - j
            b = fy - i
            
            i0 = max(i, 0);  i1 = min(i + 1, Gy - 1)
            j0 = max(j, 0);  j1 = min(j + 1, Gx - 1)
            
            v = img[y, x]
            
            T00 = luts[i0, j0, v]
            T01 = luts[i0, j1, v]
            T10 = luts[i1, j0, v]
            T11 = luts[i1, j1, v]

            T_top = (1 - a) * T00 + a * T01
            T_bot = (1 - a) * T10 + a * T11

            val = (1 - b) * T_top + b * T_bot
            out[y, x] = val
            
    return np.rint(out).astype(np.uint8)

test

In [10]:
# ===== Ô KIỂM THỬ CLAHE — dán vào 1 cell cuối notebook, chạy sau khi đã chạy các ô định nghĩa hàm =====
import numpy as np

L = 256
BIG = 10 ** 9


def global_he_lut(img):
    h = np.bincount(img.ravel(), minlength=L)
    return np.round((L - 1) * np.cumsum(h) / img.size).astype(np.uint8)


def raises(exc, fn, *a, **k):
    try:
        fn(*a, **k)
    except exc:
        return True
    except Exception:
        return False
    return False


# ---------------- clip_histogram ----------------
def t_clip_total_preserved():
    rng = np.random.default_rng(0)
    for _ in range(20):
        h = rng.integers(0, 500, size=L)
        for beta in (1, 5, 20, 100, 400):
            assert clip_histogram(h, beta).sum() == h.sum()

def t_clip_shape_nonneg():
    h = np.random.default_rng(0).integers(0, 100, size=L)
    out = clip_histogram(h, 10)
    assert out.shape == (L,) and (out >= 0).all()

def t_clip_beta_huge_unchanged():
    h = np.random.default_rng(0).integers(0, 100, size=L)
    assert np.array_equal(clip_histogram(h, BIG), h)

def t_clip_beta_equals_mean():
    h = np.zeros(L, dtype=np.int64); h[100] = 4096
    out = clip_histogram(h, 16)
    excess = 4096 - 16
    assert out.sum() == 4096
    assert out.max() <= 16 + excess // L + 1
    assert out.min() >= excess // L

def t_clip_peak_limited():
    h = np.full(L, 10, dtype=np.int64); h[50] = 1000
    out = clip_histogram(h, 20)
    excess = int(np.sum(np.maximum(h - 20, 0)))
    assert out.max() <= 20 + excess // L + 1

def t_clip_remainder():
    h = np.zeros(L, dtype=np.int64); h[0] = 1000; h[1] = 7
    assert clip_histogram(h, 3).sum() == 1007

def t_clip_no_modify_input():
    h = np.random.default_rng(0).integers(0, 100, size=L)
    c = h.copy(); clip_histogram(h, 5)
    assert np.array_equal(h, c)

def t_clip_float_beta():
    h = np.full(L, 8, dtype=np.int64); h[3] = 500
    assert clip_histogram(h, 15.7).sum() == h.sum()


# ---------------- build_tile_luts ----------------
def t_luts_shape_dtype():
    img = np.random.default_rng(0).integers(0, 256, (64, 64), dtype=np.uint8)
    luts = build_tile_luts(img, (4, 4), 2.0)
    assert luts.shape == (4, 4, L) and luts.dtype == np.uint8

def t_luts_non_square_grid():
    img = np.random.default_rng(0).integers(0, 256, (60, 90), dtype=np.uint8)
    assert build_tile_luts(img, (3, 5), 2.0).shape == (3, 5, L)

def t_luts_monotonic_255():
    img = np.random.default_rng(0).integers(0, 256, (64, 64), dtype=np.uint8)
    luts = build_tile_luts(img, (4, 4), 3.0)
    assert (np.diff(luts.astype(int), axis=-1) >= 0).all()
    assert (luts[..., -1] == 255).all()

def t_luts_one_tile_equals_he():
    img = np.random.default_rng(0).integers(0, 256, (32, 32), dtype=np.uint8)
    assert np.array_equal(build_tile_luts(img, (1, 1), BIG)[0, 0], global_he_lut(img))

def t_luts_own_histogram():
    rs = np.random.default_rng(1)
    img = np.zeros((32, 64), dtype=np.uint8)
    img[:, :32] = rs.integers(10, 60, (32, 32))
    img[:, 32:] = rs.integers(180, 230, (32, 32))
    luts = build_tile_luts(img, (1, 2), BIG)
    assert not np.array_equal(luts[0, 0], luts[0, 1])
    assert luts[0, 0, 60] == 255 and luts[0, 1, 100] == 0

def t_luts_clip1_near_identity():
    img = np.random.default_rng(2).integers(0, 256, (64, 64), dtype=np.uint8)
    lut = build_tile_luts(img, (1, 1), 1.0)[0, 0].astype(int)
    assert np.abs(lut - np.arange(L)).max() <= 8

def t_luts_higher_clip_steeper():
    rs = np.random.default_rng(3)
    img = np.clip(128 + rs.integers(-2, 3, (64, 64)), 0, 255).astype(np.uint8)
    lo = build_tile_luts(img, (1, 1), 1.0)[0, 0].astype(int)
    hi = build_tile_luts(img, (1, 1), 8.0)[0, 0].astype(int)
    assert (hi[130] - hi[126]) > (lo[130] - lo[126])

def t_luts_grid_too_big():
    assert raises(ValueError, build_tile_luts, np.zeros((4, 4), np.uint8), (8, 8), 2.0)

def t_luts_grid_invalid():
    img = np.zeros((16, 16), np.uint8)
    assert raises(ValueError, build_tile_luts, img, (0, 4), 2.0)
    assert raises(ValueError, build_tile_luts, img, (4, 0), 2.0)

def t_luts_wrong_input():
    assert raises(AssertionError, build_tile_luts, np.zeros((16, 16), np.float32))
    assert raises(AssertionError, build_tile_luts, np.zeros((16, 16, 3), np.uint8))

def t_luts_not_divisible():
    img = np.random.default_rng(0).integers(0, 256, (37, 53), dtype=np.uint8)
    assert build_tile_luts(img, (8, 8), 2.0).shape == (8, 8, L)


# ---------------- clahe ----------------
def t_clahe_shape_dtype():
    img = np.random.default_rng(0).integers(0, 256, (40, 56), dtype=np.uint8)
    out = clahe(img, (4, 4), 2.0)
    assert out.shape == img.shape and out.dtype == np.uint8

def t_clahe_one_tile_equals_he():
    img = np.random.default_rng(0).integers(0, 256, (24, 24), dtype=np.uint8)
    assert np.array_equal(clahe(img, (1, 1), BIG), global_he_lut(img)[img])

def t_clahe_constant():
    out = clahe(np.full((32, 32), 77, np.uint8), (4, 4), 2.0)
    assert (out == out[0, 0]).all()

def t_clahe_identical_tiles():
    tile = np.random.default_rng(0).integers(0, 256, (9, 9), dtype=np.uint8)
    img = np.tile(tile, (4, 4))
    luts = build_tile_luts(img, (4, 4), 2.0)
    assert all(np.array_equal(luts[0, 0], luts[i, j]) for i in range(4) for j in range(4))
    assert np.array_equal(clahe(img, (4, 4), 2.0), luts[0, 0][img])

def t_clahe_tile_centers():
    img = np.random.default_rng(0).integers(0, 256, (36, 36), dtype=np.uint8)
    luts = build_tile_luts(img, (4, 4), 2.0)
    out = clahe(img, (4, 4), 2.0)
    for i in range(4):
        for j in range(4):
            y, x = 9 * i + 4, 9 * j + 4
            assert out[y, x] == luts[i, j, img[y, x]], (i, j)

def t_clahe_corners():
    img = np.random.default_rng(0).integers(0, 256, (36, 36), dtype=np.uint8)
    luts = build_tile_luts(img, (4, 4), 2.0)
    out = clahe(img, (4, 4), 2.0)
    assert out[0, 0] == luts[0, 0, img[0, 0]]
    assert out[0, -1] == luts[0, 3, img[0, -1]]
    assert out[-1, 0] == luts[3, 0, img[-1, 0]]
    assert out[-1, -1] == luts[3, 3, img[-1, -1]]

def t_clahe_between_neighbor_luts():
    img = np.random.default_rng(0).integers(0, 256, (36, 36), dtype=np.uint8)
    luts = build_tile_luts(img, (4, 4), 2.0).astype(float)
    out = clahe(img, (4, 4), 2.0).astype(float)
    assert (out >= luts.min(axis=(0, 1))[img] - 1).all()
    assert (out <= luts.max(axis=(0, 1))[img] + 1).all()

def t_clahe_no_seams():
    img = np.tile(np.linspace(0, 255, 64).astype(np.uint8), (64, 1))
    out = clahe(img, (4, 4), 2.0).astype(int)
    assert np.abs(np.diff(out, axis=1)).max() < 40

def t_clahe_non_divisible():
    img = np.random.default_rng(0).integers(0, 256, (37, 53), dtype=np.uint8)
    assert clahe(img, (5, 7), 2.0).shape == (37, 53)

def t_clahe_local_contrast():
    rs = np.random.default_rng(4)
    img = np.zeros((48, 96), np.uint8)
    img[:, :48] = rs.integers(20, 40, (48, 48))
    img[:, 48:] = rs.integers(200, 220, (48, 48))
    out = clahe(img, (2, 2), 4.0)
    assert out[:, :40].std() > img[:, :40].std()
    assert out[:, 56:].std() > img[:, 56:].std()

def t_clahe_no_modify_input():
    img = np.random.default_rng(0).integers(0, 256, (24, 24), dtype=np.uint8)
    c = img.copy(); clahe(img, (3, 3), 2.0)
    assert np.array_equal(img, c)

def t_clahe_wrong_dtype():
    assert raises(AssertionError, clahe, np.zeros((16, 16), np.float64))


# ---------------- chạy tất cả ----------------
_tests = [(n, f) for n, f in sorted(globals().items()) if n.startswith("t_") and callable(f)]
_failed = 0
for _name, _fn in _tests:
    try:
        _fn()
        print(f"PASS  {_name}")
    except Exception as e:
        _failed += 1
        print(f"FAIL  {_name}: {type(e).__name__}: {e}")
print(f"\n{len(_tests) - _failed}/{len(_tests)} test đạt" + ("" if _failed == 0 else f" — {_failed} lỗi"))
    

PASS  t_clahe_between_neighbor_luts
PASS  t_clahe_constant
PASS  t_clahe_corners
PASS  t_clahe_identical_tiles
PASS  t_clahe_local_contrast
PASS  t_clahe_no_modify_input
PASS  t_clahe_no_seams
PASS  t_clahe_non_divisible
PASS  t_clahe_one_tile_equals_he
PASS  t_clahe_shape_dtype
PASS  t_clahe_tile_centers
PASS  t_clahe_wrong_dtype
PASS  t_clip_beta_equals_mean
PASS  t_clip_beta_huge_unchanged
PASS  t_clip_float_beta
PASS  t_clip_no_modify_input
PASS  t_clip_peak_limited
PASS  t_clip_remainder
PASS  t_clip_shape_nonneg
PASS  t_clip_total_preserved
PASS  t_luts_clip1_near_identity
PASS  t_luts_grid_invalid
PASS  t_luts_grid_too_big
PASS  t_luts_higher_clip_steeper
PASS  t_luts_monotonic_255
PASS  t_luts_non_square_grid
PASS  t_luts_not_divisible
PASS  t_luts_one_tile_equals_he
PASS  t_luts_own_histogram
PASS  t_luts_shape_dtype
PASS  t_luts_wrong_input

31/31 test đạt
